In [1]:
"""
__authors__ = Marc Vallvé Palerm, Ignasi Rebordosa Cots, Marc Salvatella Ferrerfàbrega
__date__    = Nov 2026
__title__   = Non-Linear Data Structures Project
"""

'\n__authors__ = Marc Vallvé Palerm, Ignasi Rebordosa Cots, Marc Salvatella Ferrerfàbrega\n__date__    = Nov 2026\n__title__   = Non-Linear Data Structures Project\n'

For this exercise, we decided to use an adjacency-list representation. Although a matrix-based representation would also be valid, it is less suitable for a graph with many sites and relatively few routes. For example, with 1,000 sites, a matrix would require a 1,000 × 1,000 structure, which means 1,000,000 cells and O(V²) initialization time, even though most pairs of sites are not connected. With an adjacency list, each site stores only its own data and its outgoing connections, so the structure uses O(V + E) memory, where V is the number of sites and E is the number of routes. Building the graph from the file also takes O(V + E) time on average, since a dictionary maps each site ID to its corresponding site object and allows each route endpoint to be found in O(1) average time. The adjacency list also naturally stores the connection attributes, such as destination, shipping time and restriction, and is more efficient for the traversals, topological sort and Dijkstra algorithm required later. A matrix would still be valid and would provide O(1) edge lookup, but it would waste space on the many non-existing routes.

# 2.1.1 — Data

In [2]:
#Product class
class Site:
    """Stores the data of one production site."""

    def __init__(self, site_id, product, corporation, material):
        self.id = site_id
        self.product = product
        self.corporation = corporation
        self.material = material
        self.connections = []  # List to store connections to other sites

    def get_class_info(self):
        return (
            f"Site(id={self.id}, product={self.product!r}, "
            f"corporation={self.corporation!r}, "
            f"material={self.material})"
        )

class Connection:
    """Stores the data of one connection between two production sites."""

    def __init__(self, origin, destination, shipping_time, restriction):
        self.origin = origin
        self.destination = destination
        self.shipping_time = shipping_time
        self.restriction = restriction

    def get_connection_info(self):
        return (
            f"Connection(origin={self.origin.id}, destination={self.destination.id}, "
            f"shipping_time={self.shipping_time}, restriction={self.restriction})"
        )




# 2.1.2 - Functionalities

### 2.1.2.1 - Graph Representation

In [3]:
# ============================================================
# .paed FILE FORMAT
# ============================================================

# First line: number of sites/vertices
# Number of sites

# <site_id>;<product_name>;<corporation>;<site_material> (*Number of sites)

# Next line: number of routes/edges
# Number of routes

# <origin_id>;<destination_id>;<shipping_time>;<restriction> (*Number of routes)

file_name = r"Datasets/Graphs/aCityXS.paed"

sites = [] #a list to store all the site objects
sites_by_id = {} #stores side id as primary key: site object to access them faster direct access, O(1), instead of serial access, O(n)

with open(file_name, "r") as f:
    #read first line to get number of sites
    num_sites = int(f.readline().strip())

    #initialize a list to store the sites
    for i in range(num_sites):
        #read the line and split it by semicolon
        line = f.readline().strip()
        site_id, product, corporation, material = line.split(";")
        #create a site object and append it to the list
        sites.append(Site(int(site_id), product, corporation, material))
        sites_by_id[int(site_id)] = sites[i] #store the site object in the dictionary with its id as primary key
        
    #get how many routes there are
    num_routes = int(f.readline().strip())

    for i in range(num_routes):
        #read the line and split it by semicolon
        line = f.readline().strip()

        origin_id, destination_id, shipping_time, restriction = line.split(";")
        
        #create a Connection object and append it to the list of connections of the origin site
        #----------get the site where the connection starts and ends
        origin_site = sites_by_id[int(origin_id)]
        destination_site = sites_by_id[int(destination_id)]
        origin_site.connections.append(Connection(origin_site, destination_site, int(shipping_time), restriction))

In [4]:
#represent graph as adjacency list
for site in sites:
    print(f"\nSite {site.id} ({site.product}) has connections:")
    if not site.connections:
        print(f"\t-> No connections found")
    else:
        for connection in site.connections:
            print(f"\t-> Site {connection.destination.id} ({connection.destination.product}), "
                  f"Shipping Time: {connection.shipping_time}, Restriction: {connection.restriction}")



Site 67 (Triangular prism Catun No.2) has connections:
	-> Site 103 (Copper cuboid Mk II), Shipping Time: 19, Restriction: GLASS
	-> Site 118 (Copper hemisphere edge type 3), Shipping Time: 14, Restriction: NONE

Site 87 (Iron tetrahedron Mk I) has connections:
	-> No connections found

Site 119 (Fabric tetrahedron Catun No.1) has connections:
	-> Site 103 (Copper cuboid Mk II), Shipping Time: 24, Restriction: PLASTIC
	-> Site 121 (Wood cone mid-grade salvage), Shipping Time: 13, Restriction: GLASS
	-> Site 78 (Iron hexagonal prism edge type 1), Shipping Time: 22, Restriction: ORGANIC

Site 123 (Tinted glass icosahedron rusted junk) has connections:
	-> Site 103 (Copper cuboid Mk II), Shipping Time: 12, Restriction: PLASTIC

Site 101 (Tinted glass cube Mk I) has connections:
	-> Site 77 (Wax parallelepiped Mk II), Shipping Time: 19, Restriction: GLASS
	-> Site 134 (Polystyrene hexagonal prism Mk II), Shipping Time: 11, Restriction: METAL

Site 77 (Wax parallelepiped Mk II) has connect

A -> Site ID entered by the user.

To retrieve it from our code, we will use sites_by_id[ID].

A corporation can have more than one site.

We will create a tree that starts with all the connections of A. The second level will contain the connections of A, the third level will contain the connections of those connections, and so on.

A
|-B
|_ C
...

To find all nodes reachable from A that belong to the same corporation, we will use DFS.

Since we need to look through all reachable nodes, as there may be more than one correct answer, the time complexity is O(n).

### 2.1.2.2 - Commercial Intelligence

In [5]:
ID_to_search = int(input("Enter the ID of the site to search for: "))

site_to_search = sites_by_id[ID_to_search]

visited = set() # Set to keep track of visited nodes during the search and do not repeat them
stack_corporation_nodes = [] # Stack of the result of the nodes that belong to the same corporation
corporation_to_search = site_to_search.corporation

#initialize a queue for sites waiting to be visited and append initial site (search input)
queue = []
queue.append(site_to_search)

#dict storing id primary key: site's distance from search site (depth)
depth_by_site_id = {site_to_search.id: 0}

# recursive function to find all the nodes that belong to the same corporation
def find_corporation_nodes(queue_index, corporation):

    if queue_index >= len(queue):
        return

    current_site = queue[queue_index]

    for connection in current_site.connections:
        next_site = connection.destination

        #only add site first time reached
        if next_site.id not in visited:
            visited.add(next_site.id)

            depth_by_site_id[next_site.id] = (depth_by_site_id[current_site.id] + 1)
            queue.append(next_site)

            #keep only matching corporation sites, exclude search site
            if next_site.corporation == corporation and next_site.id != ID_to_search:
                stack_corporation_nodes.append(next_site)

    #process next queued site. new sites are appended to queue to continue BFS
    find_corporation_nodes(queue_index + 1, corporation)

visited.add(site_to_search.id)
find_corporation_nodes(0, corporation_to_search)

#sort sites from deepest to shallows, using ascending ID for ties
stack_corporation_nodes.sort(key=lambda site: (-depth_by_site_id[site.id], site.id))

#display the result
print(f"Search results:")
if not stack_corporation_nodes:
    print("No other accessible sites found for this corporation.")
else:
    for site in stack_corporation_nodes:
        print(
            f"\tSite {site.id} ({site.product}) — "
            f"Corporation: {site.corporation} — "
            f"Supply-chain depth from site {ID_to_search}: "
            f"{depth_by_site_id[site.id]}"
    )

Search results:
No other accessible sites found for this corporation.


# 2.2 - Sour heart (Binary search trees)


### 2.2.2.1 - Representation as a binary search tree


In [6]:
class Inhabitant:
    def __init__(self, inhabitant_id, name, sugar_index, profession):
        self.id = inhabitant_id
        self.name = name
        self.sugar_index = sugar_index
        self.profession = profession

    def get_inhabitant_info(self):
        return (
            f"Inhabitant(id={self.id}, name={self.name!r}, "
            f"sugar_index={self.sugar_index}, profession={self.profession})"
        )

class Node:
    def __init__(self, inhabitant):
        self.inhabitant = inhabitant
        self.left = None
        self.right = None


In [7]:
#CREATE CLASS OF BINARY SEARCH TREE TO ACCESS FASTER TO ALL FUNCTIONS

class BST:
    """Binary search tree ordered by sugar index.
        if the actual sugar index is less than the current node, it goes to the left subtree, otherwise it goes to the right subtree.
        After assigning one position of the node, we go back to the root and repeat the process until all nodes are assigned.
    """

    def __init__(self):
        self.root = None
        self.nodes_by_id = {}  
        self.size = 0

    def insert(self, inhabitant):
        node = Node(inhabitant)
        self.nodes_by_id[inhabitant.id] = node

        if self.root is None: #if it is the first node (root), we assign it directly
            self.root = node
            self.size += 1
            return

        current = self.root
        while True: #else we traverse the tree to find the correct position for the new node
            if inhabitant.sugar_index < current.inhabitant.sugar_index: #if the sugar index is less than the current node, we go to the left subtree
                if current.left is None: #if the cell is empty we assign the new node to the left subtree
                    current.left = node
                    break
                current = current.left #else we iterate again to check if it needs to go left or right
            else:
                if current.right is None: #if the cell is empty we assign the new node to the right subtree
                    current.right = node
                    break
                current = current.right #else we iterate again to check if it needs to go left or right

        self.size += 1
        
    def delete(self, inhabitant_id):
        target = self.nodes_by_id.get(inhabitant_id) #get target node to get his sugar index and find it in the tree
        if target is None:
            return False  # id does not exist

        prev_node = None #save previous node to be able to delete the target node and connect the previous ones to all possible children from the actual one
        next_node = self.root

        # Find the exact node and its parent. Equal sugars go right, because insert places duplicates in the right subtree.
        while next_node is not None and next_node is not target:
            prev_node = next_node
            if target.inhabitant.sugar_index < next_node.inhabitant.sugar_index:
                next_node = next_node.left
            else:
                next_node = next_node.right

        if next_node is None: #if the node was not found, there is no node with that ID
            return False

        #1 or 0 children
        if next_node.left is None or next_node.right is None:
            
            if next_node.left is not None:
                child = next_node.left 
            elif next_node.right is not None:
                child = next_node.right
            else: #no child
                child = None
                
            if prev_node is None: #if it is the first node
                self.root = child
            elif prev_node.left is next_node: #if the child is on the left
                prev_node.left = child
            else: #if the child is on the right or there is no node (if there is no node, child will be none)
                prev_node.right = child

        # 2 children
        else:
            L = next_node.left
            R = next_node.right

            if R.left is not None: #Since all values from the right are bigger than the ones on the left, we promote R, and the left child from R will be L
                rightmost = L 
                #the right child of the last right child from L will be the left side from R 
                while rightmost.right is not None: #while we are not in the last right child from L
                    rightmost = rightmost.right
                rightmost.right = R.left

            R.left = L

            if prev_node is None: #If we delete the child
                self.root = R
            elif prev_node.left is next_node: 
                prev_node.left = R
            else:
                prev_node.right = R
        
        del self.nodes_by_id[inhabitant_id]
        self.size -= 1
        return True
            

    def inorder(self):
        #yield turns a normal function into a generator
        """Produces values one at a time in ascending sugar order instead of returning one single result. 
        If we call a function after calling it, it will be resumed from where was left off, instead of starting over. 
        This allows us to iterate through the tree without storing all the values in memory at once.
        It will go all first on the first call, then at the previous last node, it will check if there is any child, if yes go right
        This over and over, until it reaches the last node. Then it will go back to the root and check if there are any nodes left to visit.
        It only prints the lowest node found first, then the next lowest, and so on until it reaches the highest node.
        The root will always be on the middle"""
        yield from self._inorder(self.root)

    def _inorder(self, node):
        if node is not None: #Till the node is not None, we will keep going down the tree to the leftmost node, then yield the node's inhabitant, and then go to the right subtree
            yield from self._inorder(node.left) #create a recursive function that will go down the left subtree until it reaches the leftmost node, then yield the node's inhabitant, and then go to the right subtree
            yield node.inhabitant #only wen the left most node is reached, we go to this line, we output the node's inhabitant on the most right
            yield from self._inorder(node.right) #after going all left, we try to go right

    def find_nearest_element(self, sugar_index):
        node = self.root
        level = 0

        while node is not None and node.inhabitant.sugar_index != sugar_index:
            if sugar_index < node.inhabitant.sugar_index:
                node = node.left
            else:
                node = node.right
            level += 1

        #return everything as an array so we can return the level where it is and the node
        return [node, level]


In [8]:
def load_tree(filename):
    with open(filename, "r") as f:
        lines = []
        for _ in f: #store all lines on an array
            lines.append(_.strip())
        
        num_inhabitants = int(lines[0])
        tree = BST() #initialize the binary search tree

        for line in lines[1:1 + num_inhabitants]: #start at 1 line (2nd since it's an array) and end at 1 + num_inhabitants (the last line of the inhabitants)
            inhabitant_id, name, sugar_index, profession = line.split(";") #store all information
            tree.insert(Inhabitant(
                int(inhabitant_id),
                name,
                float(sugar_index.replace(",", ".")), #replace comma with dot to convert to float
                profession,
            )) #create new element and store it at the tree at the cell assigned

    return tree

tree_file = "Datasets/Trees/treeS.paed"
tree = load_tree(tree_file)

sugars = []
for inhabitant in tree.inorder():
    sugars.append(inhabitant.sugar_index)



### 2.2.2.2 - Basic Functionalities

In [9]:
PROFESSIONS = {
    "CANDY_SELLER",
    "CANDY_RECIPE_CREATOR",
    "SWEET_EVENT_ORGANIZER",
    "SWEET_GUARDIAN",
    "SWEET_AMBASSADOR",
    "SWEET_INGREDIENT_HUNTER",
    "CANDY_MAGIC_TEACHER",
    "UNEMPLOYED",
}

action = input("Add or delete an inhabitant? (A/D): ").strip().upper()

#user wants to add an inhabitant
if action == "A":
    inhabitant_id = None
    while inhabitant_id is None: #ask user for id
        try:
            inhabitant_id = int(input("Id: "))
        except ValueError:
            print("Please enter a valid integer.")

    if inhabitant_id in tree.nodes_by_id:
        print(f"Id {inhabitant_id} already exists.")
    else:
        name = input("Name: ").strip()

        sugar_index = None
        while sugar_index is None:
            try:
                sugar_index = float(input("Sugar index: ").replace(",", "."))
            except ValueError:
                print("Please enter a valid number.")

        profession = input("Profession: ").strip().upper()

        if profession not in PROFESSIONS:
            print("Invalid profession.")
        else:
            tree.insert(Inhabitant(inhabitant_id, name, sugar_index, profession)) #add new inhabitant to the binary tree
            print(f"Inhabitant {inhabitant_id} added.")

elif action == "D":
    inhabitant_id = None
    while inhabitant_id is None: #ask for id to delet
        try:
            inhabitant_id = int(input("Id to delete: "))
        except ValueError:
            print("Please enter a valid integer.")

    if tree.delete(inhabitant_id):
        print(f"Inhabitant {inhabitant_id} deleted.")
    else:
        print(f"No inhabitant with id {inhabitant_id}.")

else:
    print("Invalid option.")

Inhabitant 119 deleted.


### 2.2.2.3 - Visual representation

In [13]:
def showtree(node, spaces="", side = ""): 
    if node is None:
        return
    showtree(node.right, spaces + "    ", side = "R")
    print(side + " " + spaces + f"{node.inhabitant.id} ({node.inhabitant.sugar_index})")
    showtree(node.left, spaces + "    ", side = "L")

showtree(tree.root)

R                         1131 (94.9513)
L                             1698 (94.848)
L                                 1974 (94.7344)
R                                             1104 (94.7102)
R                                         2678 (94.5256)
L                                     1032 (94.4686)
L                                         126 (94.3436)
L                                             1998 (94.3335)
R                     1808 (94.2278)
R                                 1399 (94.1853)
R                             618 (93.8554)
R                                             366 (93.8199)
R                                         2444 (93.789)
R                                     2219 (93.6862)
L                                         2209 (93.2712)
L                                 1176 (93.1469)
R                                         464 (93.0276)
L                                     1784 (93.0091)
L                         30 (92.8904)
R                        

### 2.2.2.4 - Spy search

In [10]:
#ask user for inputs:
sugar_index = None
while sugar_index is None:
    try:
        sugar_index = float(input("Enter sugar percentage to look for: ").replace(",", "."))
    except ValueError:
        print("Please enter a valid number.")

base = input("Enter candy used to create the clone (profiterole/shortbread): ").strip().lower()
while base not in ("profiterole", "shortbread"):
    print("Invalid option: it must be profiterole or shortbread.")
    base = input("Enter candy used to create the clone (profiterole/shortbread): ").strip().lower()

# search for first inhabitant with same amount of percentage
"""Since we have the sugar index provided, we have to use the same method we did
to include new elements on the tree, if the sugar index is smaller, go left, if not go right
and the first element with the same sugar index will be the one that is nearly"""

info = tree.find_nearest_element(sugar_index)
if info[0] is None: #info[0] is the node found (None if it doesn't exist), info[1] is its level
    print("No inhabitant with that sugar index.")
else:
    print(f"Found at level {info[1]}: {info[0].inhabitant.get_inhabitant_info()}")

    #2nd search:
    #   if profiterole --> show all inhabitants in the same level in the tree as the identified node (BFS)
    #   if shortbread --> show all inhabitants in the subtree that begins at the identified node sorted based on their sugar index

    if base == "profiterole": #walk tree level by level to find level where ID is stored
        level = info[1] #depth of the identified node, already returned by find_nearest_element
        nodes_same_level = []
        waiting_nodes = [(tree.root, 0)]
        head = 0 #index of next waiting_nodes to be visited

        """We are only going to increment the size of waiting_nodes when the actual level is smaller than the target level,
        when we reach the target level, since we are going to store the nodes into a different list, we are not going to increment
        the size of the waiting_nodes"""
        while head < len(waiting_nodes): #while there's something left to process (queue grows inside the loop, therefore len(...) is not static)
            current, depth = waiting_nodes[head]
            head += 1

            if depth == level: #if the node is at the right level, append the actual node in a new list
                nodes_same_level.append(current.inhabitant)
            else: #the node is not at the target level, therefore pass to the next level
                #check if current node has 1 or 2 children
                if current.left is not None:
                    waiting_nodes.append((current.left, depth + 1)) #add new node with the next depth
                if current.right is not None:
                    waiting_nodes.append((current.right, depth + 1)) #add new node with the next depth

        for inhabitant in nodes_same_level:
            print(inhabitant.get_inhabitant_info())

    else:
        #we already have the target index (it was returned by find_nearest_element)
        #therefore we can assign as the root the target index position info[1]
        visited_nodes = []
        current = info[0]

        while current is not None or visited_nodes: #stop only when everything on the subtree was printed
            while current is not None: #go as left as possible till the last node (needed to print all his left subtree)
                visited_nodes.append(current)
                current = current.left #go 1 level left deeper 

            current = visited_nodes.pop() #remove the last index added so we can go in the other direction 
            print(current.inhabitant.get_inhabitant_info())
            current = current.right   #then its right subtree and starting again it will go all left 

Invalid option: it must be profiterole or shortbread.
No inhabitant with that sugar index.


### 2.2.2.5 - Clone capture
For this exercise we can apply pruning. The tree is a BST ordered by sugar index,
so at every node we already know where the values are, everything in its left subtree
is smaller, everything in its right subtree is greater or equal.

If a node's sugar index is smaller than or equal to the minimum, neither that node nor
anything in its left subtree can be inside the range, so we only continue to the right.
And the same on the right if a node's sugar index is greater than the maximum, 
neither that node nor anything in its right subtree can be inside the range,
so we only continue to the left.

We only visit the remaining (non-pruned) nodes, and visiting them left -> node -> right
(in-order) returns them already in ascending sugar order, so we just append the accepted
inhabitants to a list and return it without sorting anything.

In [11]:
minimum = None
while minimum is None:
    try:
        minimum = float(input("Minimum sugar index: ").replace(",", "."))
    except ValueError:
        print("Please enter a valid number.")

maximum = None
while maximum is None:
    try:
        maximum = float(input("Maximum sugar index: ").replace(",", "."))
    except ValueError:
        print("Please enter a valid number.")

if minimum > maximum: #accept them in any order
    minimum, maximum = maximum, minimum

result = []

def collect_range(node): #recursive function to get to the right branch of the tree 
    if node is None:
        return
    if node.inhabitant.sugar_index > minimum: #left subtree can still contain values >= minimum
        collect_range(node.left)
    if minimum <= node.inhabitant.sugar_index <= maximum: #the node itself is inside the range
        result.append(node.inhabitant) #add that node to the result
    if node.inhabitant.sugar_index <= maximum: #right subtree can still contain values <= maximum
        collect_range(node.right)

collect_range(tree.root)

print(f"{len(result)} inhabitants with sugar index between {minimum} and {maximum}:")
for inhabitant in result: #print all nodes 
    print(inhabitant.get_inhabitant_info())



276 inhabitants with sugar index between 55.6 and 145.7:
Inhabitant(id=2729, name='Ellie Noise Marks Greenfelder', sugar_index=55.6544, profession=UNEMPLOYED)
Inhabitant(id=1029, name='Zoltan Pepper Wilderman Zemlak', sugar_index=55.8235, profession=SWEET_INGREDIENT_HUNTER)
Inhabitant(id=1626, name='Helen Highwater Sanford Hettinger', sugar_index=56.1376, profession=CANDY_MAGIC_TEACHER)
Inhabitant(id=869, name='Vlad Tire Volkman Lindgren', sugar_index=56.2537, profession=CANDY_RECIPE_CREATOR)
Inhabitant(id=2743, name='Denny Juan Heredatt Thompson Sanford', sugar_index=56.2874, profession=SWEET_GUARDIAN)
Inhabitant(id=1961, name='Justin Sane Schmitt Parker', sugar_index=56.29, profession=SWEET_AMBASSADOR)
Inhabitant(id=554, name='I. Ron Stomach Cole Runte', sugar_index=56.3683, profession=SWEET_GUARDIAN)
Inhabitant(id=890, name='Lance Lyde Rodriguez Yost', sugar_index=56.5074, profession=SWEET_AMBASSADOR)
Inhabitant(id=747, name='Reid Enright Gleason Kemmer', sugar_index=56.6503, profes

### 2.2.2.6 - Infection origin

# 2.3 Inventory breakers (Tables)

## 2.3.1 Data

## 2.3.2 Functionalities

### 2.3.2.1 Representation as tables

In [ ]:
VALID_COLORS = {"GREEN", "YELLOW", "RED", "BLUE", "PINK", "ORANGE"}


def load_inventory(filename):
    inventory = {}  #dictionary
    with open(filename, "r") as f:
        count = int(f.readline().strip())  #read first line and convert it to number of inventions

        for line_number in range(2,
                                 count + 2):  #loop once per invention, first invention is on line 2, 3 fields per invention
            line = f.readline().strip()  #read line
            name, status_text, color = line.split(";")  #separate line into its 3 fields, separated by semicolons

            #remove whitespace around fields and convert to desired case
            name = name.strip()
            status_text = status_text.strip().lower()
            color = color.strip().upper()

            if status_text not in ("true", "false"):
                raise ValueError(f"Invalid status on line {line_number}")  #stop with error if status is invalid
            if color not in VALID_COLORS:
                raise ValueError(f"Invalid color on line {line_number}")
            if name in inventory:  #check duplicates
                raise ValueError(f"Duplicate invention name: {name}")

            inventory[name] = {  #add invention in dictionary
                "status": status_text == "true",  #convert to boolean
                "color": color,  #store normalized color string
            }
    return inventory


inventory = load_inventory(r"Datasets/Tables/tablesS.paed")

In [ ]:
def add_invention(inventory, name, status, color):
    name = name.strip()  #cleans received names from whitespaces
    color = color.strip().upper()

    if not name:  #check if name is empty
        raise ValueError("You must name the invention")
    if name in inventory:
        raise ValueError("This invention already exists")
    if color not in VALID_COLORS:
        raise ValueError("Unknown color")

    inventory[name] = {"status": status, "color": color}


def delete_invention(inventory, name):
    if name not in inventory:
        return False
    del inventory[name]  #remove entry in dictionary (inventory) with that name
    return True


def get_invention(inventory, name):  #look invention by name
    return inventory.get(name)  #return status and color, None if name is not found


def inventions_with_prefix(inventory, prefix):
    prefix = prefix.strip().casefold()
    return [  #build and return a list with every matching name and its data
        (name, data)
        for name, data in inventory.items()
        if name.casefold().startswith(prefix)
    ]


def broken_color_percentages(inventory):
    broken = [  #list of broken inventions
        data
        for data in inventory.values()
        if data["status"] is False  #keep invention when status is False (broken)
    ]

    if not broken:
        return {}

    counts = {color: 0 for color in VALID_COLORS}  #start each allowed color count at 0
    for data in broken:
        counts[data["color"]] += 1  #add 1 to the count of broken inventions to that color

    total = len(broken)
    return {
        color: count * 100 / total
        for color, count in counts.items()  #examine each color and its count
        if count > 0  #include only colors that are among broken inventions
    }


In [ ]:
while True:
    print("\nInventory menu")
    print("1. Add invention")
    print("2. Delete an invention")
    print("3. Look up an invention")
    print("4. Search by name prefix")
    print("5. Show broken-invention color statistics")
    print("0. Exit")

    choice = input("Choose an option: ").strip()

    if choice == "1":
        name = input("Invention name: ")
        status_text = input("Is it working? (yes/no): ").strip().lower()
        color = input("Color: ").strip().upper()

        if status_text not in ("yes", "no"):
            print("Please, enter yes or no.")
        else:
            add_invention(inventory, name, status_text == "yes", color)
            print("Invention added!")

    elif choice == "2":
        name = input("Name of the invention to delete: ")
        if delete_invention(inventory, name):
            print("Invention deleted!")
        else:
            print("Inventor not found!")

    elif choice == "3":
        name = input("Enter the invention to look up: ")
        invention = get_invention(inventory, name)
        if invention is None:
            print("No invention with that name was found!")
        else:
            print(f"Status: {'Functioning' if invention['status'] else 'Broken'}")
            print(f"Color: {invention['color']}")

    elif choice == "4":
        prefix = input("Enter the beginning of the invention to look up: ")
        matches = inventions_with_prefix(inventory, prefix)
        if matches:
            for name, invention in matches:
                status = "Working" if invention["status"] else "Broken"
                print(f"{name} - {status}, {invention['color']}")

        else:
            print("No inventions found with that prefix.")

    elif choice == "5":
        percentages = broken_color_percentages(inventory)
        if percentages:
            for color, percentage in percentages.items():
                print(f"{color} - {percentage:.1f}%")
        else:
            print("There are no broken inventions.")

    elif choice == "0":
        print("Goodbye!")
        break

    else:
        print("Choose one of the listed options.")